# Comp1 - Traditional radiomics

Mainly suited to modeling and characterization with traditional radiomics. A typical use case is to investigate the role of the rad_score in the final clinical diagnosis.

The general form of the data is as follows (specific files and folder names may differ):
1. The `images` folder stores all CT, MRI and other data of the study subjects.
2. The `masks` folder stores the manually delineated ROI regions, corresponding to the files in the images folder.
3. The `label.txt` file holds the label of each patient, e.g. tumor benign/malignant status, 5-year survival status, etc.

## Onekey steps

1. Data validation: check whether the data format is correct.
2. Radiomics feature extraction: if the data passes the first-step check, extract the features of the corresponding data.
3. Read the annotation data information.
4. Concatenate features with annotation data to form a dataset.
5. Inspect some statistics and check whether the data contains outliers.
6. Normalization: transform the data to follow N~(0, 1).
7. Select features using correlation coefficients such as spearman, pearson, etc.
8. Build the training and test sets. Here a random split is used; for proper multi-center validation you should build two datasets according to your own scenario.
9. Select features with Lasso, keeping the non-zero terms as features for the subsequent model.
10. Use machine learning algorithms such as LR, SVM, RF, etc. for the learning task.
11. Visualize model results, e.g. AUC, ROC curve, confusion matrix, etc.


### Specify data

This module has 3 parameters you need to define yourself

1. `mydir`: the path where the data is stored.
2. `labelf`: the annotation file for each sample.
3. `labels`: the targets for the AI system to learn, e.g. tumor benign/malignant status, T-stage, etc.

In [ ]:
import os
import pandas as pd
from IPython.display import display
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd

os.makedirs('img', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('features', exist_ok=True)

# Set the task prefix
task_type = 'Radiomics_'
# Set the data directory
# mydir = r'path to your own data'
mydir = get_param_in_cwd('radio_dir')
if mydir == okds.ct:
    print(f'Using Onekey data: {okds.ct}; if this is not as expected, please change the directory location!')
# Corresponding label file
group_info = get_param_in_cwd('dataset_column') or 'group'
labelf = get_param_in_cwd('label_file') or os.path.join(mydir, 'label.csv')
# Read the column names of the label data
labels = [get_param_in_cwd('task_column') or 'label']

### Matching images and masks

The filenames in the images and masks folders must correspond one-to-one. e.g. `1.nii.gz` is a file in images, and a `1.nii.gz` file must also exist in the masks folder.

You can of course use a custom function to obtain and parse the data.

In [ ]:
from pathlib import Path
from onekey_algo.custom.components.Radiology import diagnose_3d_image_mask_settings, get_image_mask_from_dir

# Generate image-mask pairs in a one-to-one relationship. You can also replace this with a custom function.
images, masks = get_image_mask_from_dir(mydir, images=f'images', masks=f'masks')
# diagnose_3d_image_mask_settings(images, masks, verbose=True, assume_masks=[0, 1])
print(f'Obtained {len(images)} samples.')

# Traditional radiomics features

Use pyradiomics to extract traditional radiomics features. Normally this part does not need modification; below is the specific Onekey wrapper interface.

```python
def extract(self, images: Union[str, List[str]], 
            masks: Union[str, List[str]], labels: Union[int, List[int]] = 1, settings=None)
"""
    * images: List, the list of images to extract from.
    * masks: List, the masks corresponding to the images to extract from; must correspond one-to-one with Images.
    * labels: which annotation label to extract features for. Defaults to extracting label=1.
    * settings: other feature-extraction parameters. Defaults to None.

"""
```

```python
def get_label_data_frame(self, label: int = 1, column_names=None, images='images', masks='labels')
"""
    * label: get features for the corresponding label.
    * columns_names: defaults to None; use the program-defined column names.
"""
```
    
```python
def get_image_mask_from_dir(root, images='images', masks='labels')
"""
    * root: the directory from which to extract features.
    * images: the folder name of the raw data inside the root directory.
    * masks: the folder name of the annotation data inside the root directory.
"""
```


In [ ]:
import warnings
import pandas as pd
 
warnings.filterwarnings("ignore")

from onekey_algo.custom.components.Radiology import ConventionalRadiomics

rad_ = None
if os.path.exists(f'features/rad_features.csv'):
    rad_data = pd.read_csv(f'features/rad_features.csv', header=0)
else:
    images, masks = get_image_mask_from_dir(mydir, images=f'images', masks=f'masks')
    # If you want to customize feature extraction, you can use param_file.
    param_file = get_param_in_cwd('extractor_settings')
    radiomics = ConventionalRadiomics(param_file, correctMask=True)
    radiomics.extract(images, masks, workers=1, with_fd=get_param_in_cwd('with_fd'))
    rad_data = radiomics.get_label_data_frame(label=1)
    rad_data.columns = [f"intra_{c.replace('-', '_')}" if c != 'ID' else c for c in rad_data.columns]
    rad_data.to_csv(f'features/rad_features.csv', header=True, index=False)
if rad_ is None:
    rad_ = rad_data
else:
    rad_ = pd.merge(rad_, rad_data, on='ID', how='inner')
rad_data = rad_
rad_data

## Feature statistics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sorted_counts = pd.DataFrame([c.split('_')[-2] for c in rad_data.columns if c !='ID']).value_counts()
sorted_counts = pd.DataFrame(sorted_counts, columns=['count']).reset_index()
sorted_counts = sorted_counts.sort_values(0)
sorted_counts.columns = ['feature_group', 'count']
display(sorted_counts)

plt.figure(figsize=(20, 10))
ax = plt.subplot(121)
plt.pie(sorted_counts['count'], labels=[i for i in sorted_counts['feature_group']], startangle=0,
        counterclock = False, autopct = '%.1f%%')
# plt.bar_label(bar.containers[0])
ax = plt.subplot(122)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
bar = sns.barplot(data=sorted_counts, x='feature_group', y='count', )
plt.savefig(f'img/{task_type}feature_ratio.svg', bbox_inches = 'tight')

## Annotation data

The data is stored in csv format. If your data is in another format, you can use a custom function to read the result for each sample.

label_data must be a `DataFrame` containing an ID column followed by the labels columns; it can have multiple columns and supports Multi-Task.

In [ ]:
from onekey_algo.custom.components.comp1 import fillna
label_data = pd.read_csv(labelf, dtype={'ID': str})
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
label_data

## Feature concatenation 

Merge the annotation data `label_data` with `rad_data` to obtain the training data.

**Note:** 
1. You need to drop the ID column
2. If you find that some data is missing, check whether the data matches.

In [ ]:
from onekey_algo.custom.utils import print_join_info
event_col = get_param_in_cwd('event_col')
duration_col= get_param_in_cwd('duration_col')

print_join_info(rad_data, label_data)
combined_data = pd.merge(rad_data, label_data[['ID', event_col, duration_col, 'group']], on=['ID'], how='inner')
print(combined_data[event_col].value_counts())
combined_data

## Normalization

`normalize_df` is the normalization API in onekey, transforming the data to zero mean and unit variance. The normalization method is

$column = \frac{column - mean}{std}$

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df
data = normalize_df(combined_data, not_norm=['ID', event_col, duration_col], group='group', use_train=True)
data = data.dropna(axis=1)
data.describe()

In [ ]:
from onekey_algo.custom.components.comp1 import normalize_df, select_feature

corr_name = get_param_in_cwd('corr_name', 'pearson')
if os.path.exists(f'features/{task_type}rad_features_sel.csv') and False:
    data = pd.read_csv(f'features/{task_type}rad_features_sel.csv', header=0)
else:
    tgroup = data[data['group'] == 'train']
    sel_feature = select_feature(tgroup[[c for c in tgroup.columns if c not in [event_col, duration_col]]].corr(corr_name), 
                                 threshold=0.9, topn=2048, verbose=False)
    data = data[['ID'] + sel_feature + [event_col, duration_col, 'group']]
    data.to_csv(f'features/{task_type}rad_features_sel.csv', header=True, index=False)
data

## Build the dataset

Separate the training data X from the supervision information y, and split the training data; the usual split ratio is 80%-20%

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp
from collections import OrderedDict


train_data = data[(data[group_info] == 'train')]

# subsets = [s for s in label_data['group'].value_counts().index if s != 'train']
subsets = get_param_in_cwd('subsets')
val_datasets = OrderedDict()
for subset in subsets:
    val_data = data[data[group_info] == subset]
    val_datasets[subset] = val_data
    val_data.to_csv(f'features/{task_type}{subset}_features_norm.csv', index=False)

print(', '.join([f"{subset} sample count: {d_.shape}" for subset, d_ in val_datasets.items()]))

# Univariable Cox

If you think there are too many features

**verbose**: defaults to False, does not output logs.

**topk**: you can specify how many top-k features to keep.

In [ ]:
from onekey_algo.custom.components.survival import uni_cox

if os.path.exists(f'features/{task_type}rad_features_unisel.csv') and False:
    train_data = pd.read_csv(f'features/{task_type}rad_features_unisel.csv')
else:
    sel_features = uni_cox(train_data, duration_col=duration_col, event_col=event_col,
                           cols=[c for c in train_data.columns if c not in [event_col, duration_col, 'ID', 'group']], 
                           verbose=False, topk=25,pvalue_thres=get_param_in_cwd('pvalue', 0.05))
    train_data = train_data[['ID'] + sel_features + [event_col, duration_col, 'group']]
    train_data.to_csv(f'features/{task_type}rad_features_unisel.csv', header=True, index=False)
train_data

# Cox-Lasso feature selection

In [ ]:
from onekey_algo.custom.components.survival import get_x_y_survival, lasso_cox_cv
COEF_THRESHOLD = 1e-6

X, y = get_x_y_survival(train_data, val_outcome=1, event_col=event_col, duration_col=duration_col)
sel_features = lasso_cox_cv(X, y, max_iter=100,  norm_X=False, prefix=f"{task_type}", l1_ratio=0.5, cv=10, weights_fig_size=(10, 15))

### Select features

Use the features after Lasso-Cox.

In [ ]:
train_data = train_data[['ID'] + list(sel_features.index) + [event_col, duration_col]]
for subset in subsets:
    val_datasets[subset] = val_datasets[subset][['ID'] + list(sel_features.index) + [event_col, duration_col]]
    val_datasets[subset].to_csv(f'features/{task_type}{subset}_cox.csv', index=False)

### Cluster analysis

By changing the variable name, you can visualize the cluster-analysis matrix under different correlation coefficients.

Note: when there are very many features (more than 100), try not to visualize, otherwise it will take very long to run.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

if train_data.shape[1] < 150:
    pp = sns.clustermap(train_data[[c for c in train_data.columns if c not in [event_col, duration_col]]].corr(corr_name), 
                        linewidths=.5, figsize=(20.0, 16.0), cmap='YlGnBu')
    plt.setp(pp.ax_heatmap.get_yticklabels(), rotation=0)
    plt.savefig(f'img/{task_type}feature_cluster.svg', bbox_inches = 'tight')

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter(penalizer=0.1)
cph.fit(train_data[[c for c in train_data.columns if c != 'ID']], duration_col=duration_col, event_col=event_col)
cph.print_summary()

In [ ]:
print(cph.concordance_index_)
su = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
su.columns = ['HR', 'HR lower 95%', 'HR upper 95%', 'pvalue']
su.reset_index().to_csv(f'features/{task_type}features_HR.csv', index=False)
su

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
cph.plot(hazard_ratios=True)
plt.savefig(f'img/{task_type}feature_pvalue.svg')
plt.show()

In [ ]:
# Determine the risk-group cutoff as the training-set median of the predicted partial hazard.
# The cutoff is fixed in the training set and applied unchanged to all cohorts,
# as in the manuscript; no outcome-optimized cutoff is used.
features = cph.params_.index.tolist()
train_ph = cph.predict_partial_hazard(train_data[features])
median_threshold = float(train_ph.median())
print(f"Training-set median cutoff: {median_threshold:.4f}")

bst_split = {subset: median_threshold for subset in val_datasets.keys()}
bst_split['train'] = median_threshold

In [ ]:
# Apply the training-set median cutoff to all cohorts and plot KM curves.

from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
import matplotlib.pyplot as plt
import pandas as pd

# 1. Set the configuration
thres = 1e-4
# The training-set median cutoff (median_threshold, previous cell) is applied unchanged to all cohorts
bst_split = {'train': median_threshold, 'test1': median_threshold, 'test2': median_threshold}
loc = {'train': 0.2, 'test1': 0.5, 'test2': 0.5}

# Get the feature column names used in model training, avoiding the ID, time or event columns
feature_cols = cph.params_.index.tolist()

for subset, test_data in val_datasets.items():
    if test_data.empty:
        continue
        
    # --- A. Compute C-index ---
    # The score method needs a full DataFrame containing features and survival information
    # Ensure test_data contains duration_col and event_col
    c_index = cph.score(test_data[feature_cols + [duration_col, event_col]], 
                        scoring_method="concordance_index")

    # --- B. Prediction and grouping ---
    # Explicitly rename the prediction column to 'PH' to prevent KeyError: 0
    y_pred = cph.predict_partial_hazard(test_data[feature_cols]).rename('PH')
    cox_data = pd.concat([test_data, y_pred], axis=1)
    
    # Group using the training-set median threshold
    current_threshold = bst_split.get(subset, median_threshold)
    cox_data['HR'] = cox_data['PH'] > current_threshold

    # --- C. Statistical test ---
    dem = (cox_data["HR"] == True)
    # Ensure both groups have data to prevent logrank_test errors
    if dem.any() and (~dem).any():
        results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                               event_observed_A=cox_data[event_col][dem], 
                               event_observed_B=cox_data[event_col][~dem])
        p_value = f"={results.p_value:.3f}" if results.p_value > thres else f'<{thres}'
    else:
        p_value = "N/A"

    # --- D. Plotting ---
    plt.figure(figsize=(8, 6))
    plt.title(f"Cohort {subset} C-index: {c_index:.3f}")
    plt.ylabel('Probability')
    
    kmf_high = KaplanMeierFitter()
    kmf_low = KaplanMeierFitter()
    
    if dem.any():
        kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], label="High Risk")
        kmf_high.plot_survival_function(color='r')
        
    if (~dem).any():
        kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], label="Low Risk")
        kmf_low.plot_survival_function(color='g')
    
    plt.text(0.5, loc.get(subset, 0.2), f"P{p_value}")
    plt.xlabel('Time(months)')
    
    # add_at_risk_counts works best only when both groups exist
    try:
        add_at_risk_counts(kmf_high, kmf_low, rows_to_show=['At risk'])
    except:
        pass

    plt.savefig(f'img/{task_type}KM_{subset}.svg', bbox_inches='tight')
    plt.show()

In [ ]:
import os
import numpy as np

def get_prediction(model: CoxPHFitter, data, ID=None, **kwargs):
    hr = model.predict_partial_hazard(data)
    expectation = model.predict_expectation(data)
    
    predictions = pd.concat([hr, expectation], axis=1)
    predictions.columns = ['HR', 'expectation']
    if ID is not None:
        predictions = pd.concat([ID, hr, expectation], axis=1)
        predictions.columns = ['ID', 'HR', 'expectation']
    else:
        predictions = pd.concat([hr, expectation], axis=1)
        predictions.columns = ['HR', 'expectation']
    return predictions

os.makedirs('results', exist_ok=True)
info = []
for subset, test_data in val_datasets.items():
    if subset in get_param_in_cwd('subsets'):
        results = get_prediction(cph, test_data, ID=test_data['ID'])
        results.to_csv(f'results/{task_type}cox_predictions_{subset}.csv', index=False)
        results['group'] = subset
        info.append(results)
        pd.merge(results, label_data[['ID', event_col, duration_col]], on='ID', how='inner').to_csv(f'features/{task_type}4xtile_{subset}.txt', 
                                                                                                    index=False, sep='\t')
info = pd.concat(info, axis=0)
info